# 03. VLM 판단을 ROS 행동으로 연결 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

모델 판단은 이동 제안입니다. 최종 행동은 관측의 신선도와 거북이 상태까지 검사한 제어기가 결정합니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab03_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![VLM 판단을 ROS 행동으로 연결의 입력·처리·출력](../교재그림/guided-vla/day4-03.svg)

ROS 노드는 영상 추론 worker와 제어 timer를 분리합니다. worker가 관측·판단·발급 시각을 갱신하고 timer는 같은 승인에 대해 follow_motion을 반복 호출합니다. 최종 command뿐 아니라 reason을 읽어 모델 STOP, 검사 거부, 목표 도착을 구분합니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![4일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day4-system.svg)

터미널 A/B/C는 실행·관찰 역할이고 ROS 노드와 HTTP 서비스는 실제 처리 주체입니다. 02는 파일 요청만, 05는 독립 YOLO 실행만 다루므로 매 단원에서 모든 상자를 동시에 실행하지 않습니다. 아래 단원별 도식과 함께 현재 경로를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 비동기 작업과 ROS callback

- callback은 수신 시 실행되는 짧은 처리 함수입니다.
- worker는 느린 MediaPipe·HTTP 요청을 담당합니다.
- Future는 진행 상태이며 완료 전 결과를 기다리지 않습니다.
- 50ms timer는 새 추론 없이도 현재 승인을 검사합니다.

ROS executor는 수신 callback과 timer callback을 실행합니다. on_image 안에서 느린 HTTP 요청이 끝날 때까지 기다리면 다른 callback 처리가 늦어져 Pose나 정지 판단도 지연될 수 있습니다. 이 코드에서는 on_image가 최신 입력만 교체하고 ThreadPoolExecutor의 worker 하나가 process를 실행합니다.

Future는 나중에 완료될 작업의 핸들입니다. tick은 future.done()이 참일 때만 result()로 반환값을 가져옵니다. 따라서 모델이 처리 중이어도 기존 승인에 대한 시간 검사와 속도 발행은 계속할 수 있습니다. worker 하나라는 말은 모델 요청을 무제한 병렬로 보내지 않는다는 뜻이지 시스템 전체가 한 스레드로만 동작한다는 뜻은 아닙니다.

20Hz는 제어 timer의 목표 호출 주기입니다. VLM이 초당 20회 판단한다는 뜻도, 실제 시스템의 모든 tick 간격이 정확히 50ms로 보장된다는 뜻도 아닙니다. 추론 빈도와 제어 검사 빈도는 로그의 judgment와 action을 구분해 확인합니다.


### 2. 실행 아키텍처: 프레임 소유권을 유지하기

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day4-03.svg)

- on_image → latest; 빈 worker만 다음 입력을 가져갑니다.
- process는 동일 영상에서 좌표와 VLM 판단을 만듭니다.
- observation·decision·issued_ms·control_image를 함께 회수합니다.
- timer → follow_motion → Twist와 /vla/action

latest는 앞으로 처리할 입력이고 observation은 처리가 끝나 현재 제어에 쓰는 관측입니다. 둘은 서로 다른 시점일 수 있습니다. process는 선택한 영상에서 MediaPipe 좌표를 얻고 그 원본 영상의 JPEG를 VLM에 보냅니다. rule 모드라면 같은 위치에서 규칙 판정을 사용합니다. VLM 모드에서는 유효한 한 손 중심이 없으면 요청을 생략하고 STOP으로 연결합니다.

작업이 완료되면 tick은 관측·판단·issued_ms·원문·지연·제어용 영상을 한 묶음으로 회수합니다. 새 결과를 회수한 뒤 worker가 비었으면 최신 대기 입력을 제출하고, 그 뒤 follow_motion을 호출합니다. 이 순서가 코드의 핵심입니다. 오래된 판단에 최신 카메라 좌표를 붙이지 않고 모델이 실제 본 프레임을 제어의 근거로 유지합니다.

overlay도 control_image를 우선 표시합니다. 카메라 미리보기보다 화면이 늦어 보일 수 있지만 표시된 목표와 판단의 근거를 일치시키기 위한 동작입니다. 이 화면을 무조건 최신 프레임으로 바꾸면 지연이 사라진 것처럼 보이지만 제어 근거는 달라지지 않습니다.


### 3. 코드 추적: FOLLOW가 STOP이 되는 경계

- 촬영 age → 승인 TTL → pose age를 검사합니다.
- 판정 형식 → frame 일치 → 유효한 손 중심을 검사합니다.
- 목표 도착·방향 오차를 반영해 최종 속도를 만듭니다.
- 모든 거부 분기는 reason과 0 속도를 반환합니다.

follow_motion의 입력에는 observation과 decision뿐 아니라 pose, now_ms, issued_ms, pose_ms가 들어갑니다. 모델이 과거 이미지에서 FOLLOW를 말했더라도 지금 그 관측이 오래됐거나 거북이 위치를 모르거나 승인이 만료되면 이동하지 않습니다. missing_input, stale_or_future_frame, expired, stale_pose 등의 reason은 어느 경계에서 거부되었는지 알려 줍니다.

검사는 코드에 쓰인 순서대로 진행됩니다. 여러 문제가 동시에 있으면 처음 걸린 reason이 반환됩니다. 예를 들어 판정 frame도 잘못되고 TTL도 만료라면 expired가 먼저 보일 수 있습니다. 합성 비교에서 조건을 하나씩만 바꾸는 이유가 여기에 있습니다. 각 행은 특정 분기를 따로 드러내도록 설계되어 있습니다.

VLM 모드에서 손 모양 의미는 VLM 판정이 맡습니다. MediaPipe의 규칙 state가 UNKNOWN이더라도 유효한 한 손 중심이 있고 VLM이 정상 NON_FIST/FOLLOW를 반환하면 이후 검사를 통과할 수 있습니다. 단순히 rule state까지 무조건 다시 강제하면 모드별 판단 주체를 바꾸는 수정이 됩니다.


### 4. 로그와 코드 수정: 판단과 행동을 따로 읽기

- judgment는 모델 결과, action은 매 tick의 최종 제어입니다.
- rule/vlm 비교에서는 좌표 변환과 시간 조건을 유지합니다.
- issued_ms는 새 결과를 받을 때만 교체합니다.
- worker 오류는 기존 이동 근거를 비우고 정지로 연결합니다.

JSONL의 kind=session은 설정, judgment는 한 번 완료된 인식·판단, action은 특정 시각의 최종 속도, error는 처리 오류입니다. judgment 한 행 이후 action 여러 행이 있는 것이 정상입니다. 같은 frame_id라도 시간이 지나면서 tracking에서 expired로 바뀔 수 있습니다. 이를 모델이 말을 바꾼 것으로 해석하지 않습니다.

직접 수정한다면 ROS 연결과 worker 생명주기는 nepes_vla_demo/hand_follow.py, 검증과 속도 계산은 nepes_vla_core/hand_follow.py에서 찾습니다. common.hand_follow는 노트북이 같은 core를 읽도록 노출하는 경로입니다. 파일 이름이 같아도 책임이 다르므로 어느 파일을 바꿨는지 확인합니다.

issued_ms를 tick마다 현재 시각으로 덮으면 새 판단이 없어도 TTL이 계속 연장됩니다. frame 검사를 제거하면 과거 판단이 현재 좌표를 승인합니다. 이런 수정은 단순 성능 개선이 아니라 제어 의미의 변경입니다. 먼저 같은 합성 입력으로 정상·거부 분기가 유지되는지 확인한 뒤 실제 rule/vlm 실행을 비교합니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [ros_ws/src/nepes_vla_demo/nepes_vla_demo/hand_follow.py](ros_ws/src/nepes_vla_demo/nepes_vla_demo/hand_follow.py) |
| 바꾸어 볼 조건 | `mode:=rule / vlm` |
| 예상되는 차이 | 고정 관측에 frame 불일치·승인 만료·pose 갱신 중단을 하나씩 넣습니다. 이후 동일 좌표·TTL에서 rule과 vlm의 움직임을 비교합니다. |
| 직접 구현·수정할 때 | 모드를 바꾸기 전에 이전 제어기를 종료합니다. frame_id 검사를 제거하면 과거 판단을 새 좌표에 붙일 수 있습니다. 20Hz timer가 새 승인을 만들지는 않으며 issued_ms를 매 tick 갱신하면 안 됩니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 3. 준비 — 실제 실행 코드와 같은 함수를 사용합니다

아래 CPU 예제는 카메라·모델·ROS 없이 자료의 흐름을 분리해 보여 줍니다. 통과가 실제 인식 성공이나 장비 성능을 뜻하지는 않습니다.


In [ ]:
from pathlib import Path
import sys, json, inspect, uuid
from datetime import datetime, timezone
from IPython.display import display, Markdown, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'09.Pseudo-VLA_v2').is_dir())
LESSON = ROOT/'09.Pseudo-VLA_v2'
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
# 매 단원 새 커널: 다른 일차의 common 모듈이 남아 있지 않게 합니다.
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = ROOT/'Data/outputs/09.Pseudo-VLA_v2/guided'/RUN

# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
def show_rows(rows):
    """조건·결과를 한 표에서 읽습니다. 수강생이 답안 dict를 채우지 않습니다."""
    if not rows:
        print('표시할 결과 없음')
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    def value(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    lines = [' | '.join(keys), ' | '.join(['---']*len(keys))]
    lines += [' | '.join(value(row.get(key, '')) for key in keys) for row in rows]
    display(Markdown('\n'.join(lines)))

# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
def save_demo(name, rows):
    """합성 예제임을 파일에도 명시합니다. 모델 측정값이나 수강생 답안이 아닙니다."""
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT/(name+'.json')
    path.write_text(json.dumps({'kind':'synthetic-guided-demo', 'model_measured':False,
                               'rows':rows}, ensure_ascii=False, indent=2)+'\n')
    print('자동 저장:', path)
print('실행 폴더:', LESSON)


### 완성 코드 읽기

아래 셀은 실행에 쓰는 함수 본문을 그대로 표시합니다. **입력 검사 → 조건 분기 → 결과 반환**을 순서대로 읽습니다. 노트북에 복사한 별도 정답과 실제 동작이 어긋나지 않도록 같은 소스를 조회합니다.


In [ ]:
from common.hand_follow import follow_motion
# 함수의 실제 파일과 주석을 함께 표시합니다. 이 셀은 소스를 수정하지 않습니다.
for function in (follow_motion,):
    print(inspect.getsourcefile(function))
    display(Code(inspect.getsource(function), language='python'))


### 한 줄씩 읽는 처리 과정

ROS 노드는 영상 추론 worker와 제어 timer를 분리합니다. worker가 관측·판단·발급 시각을 갱신하고 timer는 같은 승인에 대해 follow_motion을 반복 호출합니다. 최종 command뿐 아니라 reason을 읽어 모델 STOP, 검사 거부, 목표 도착을 구분합니다.

모드를 바꾸기 전에 이전 제어기를 종료합니다. frame_id 검사를 제거하면 과거 판단을 새 좌표에 붙일 수 있습니다. 20Hz timer가 새 승인을 만들지는 않으며 issued_ms를 매 tick 갱신하면 안 됩니다.


## 4. 조건을 바꾸어 실행하고 표로 비교하기

고정 관측에 frame 불일치·승인 만료·pose 갱신 중단을 하나씩 넣습니다. 이후 동일 좌표·TTL에서 rule과 vlm의 움직임을 비교합니다.

아래 입력은 경계 조건을 설명하기 위한 고정 예제입니다. 주석을 읽고 셀 전체를 실행하면 비교표와 검사 결과가 자동으로 나옵니다.


In [ ]:
from common.hand_follow import follow_motion, rule_decision
observation = dict(frame_id='demo-1', captured_ms=1000, center=[1, .5], state='NON_FIST')
decision = rule_decision(observation)
base = dict(observation=observation, decision=decision, pose=(5.5, 5.5, 0),
            now_ms=1100, issued_ms=1050, pose_ms=1100)
# 기준 입력을 보존하고 각 행에서 원인 하나만 바꿉니다. **는 변경 필드를 덮어씁니다.
cases = [
    ('정상', {}),
    ('frame 불일치', {'decision': {**decision, 'frame_id': 'other'}}),
    ('승인 만료', {'now_ms': 1450}),
    ('pose 갱신 중단', {'pose_ms': 500}),
    ('목표 도착', {'pose': (10, 5.5, 0)}),
    ('판단 없음', {'decision': None}),
]
rows = [dict(case=name, **follow_motion(**{**base, **changes})) for name, changes in cases]
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert rows[0]['command'] == 'FOLLOW'
assert all(row['command'] == 'STOP' for row in rows[1:])
assert rows[1]['reason'] == 'wrong_frame'
assert rows[2]['reason'] == 'expired'
# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
save_demo('approval-cases', rows)


## 5. 실제 입력 실행 — 화면과 로그 연결

본수업은 로컬 카메라 화면을 함께 보며 역할을 교대합니다. 아래의 A/B/C는 터미널 역할입니다. 카메라와 제어기는 각각 하나만 실행합니다. 실제 실행의 지연과 인식 결과는 위 합성 예제와 별도로 확인합니다.


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


새 Terminal에서는 아래 블록의 첫 줄 bash부터 전체를 복사합니다. 로컬·원격 모두 오프라인 수기 서명 명부의 본인 번호를 ROS_DOMAIN_ID로 사용합니다. 예시 7을 본인 번호로 바꾸고 본인 A·B·C 터미널에 같은 값을 적용합니다. 공동 통신 실습에서만 강사가 지정한 동일 명부 번호를 사용합니다. 노트북 Python 셀에 붙이지 않습니다. 이미 환경을 설정한 같은 B에서 모드·TTL만 바꿀 때는 Ctrl+C 후 launch 명령만 실행합니다.

**처음 준비 또는 패키지를 찾지 못할 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA_v2/scripts/build_workspace.sh
```

**A — 카메라 입력:** 이미 A가 실행 중이면 그대로 유지합니다. 카메라 publisher는 하나만 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

카메라 번호는 배정값입니다. 녹화 입력으로 비교할 때는 A를 Ctrl+C로 종료하고 같은 A에서 아래 명령을 실행합니다. 매 조건에서 같은 시작 지점으로 재생을 다시 시작합니다.
```bash
ros2 run nepes_ros_lab recorded_image_publisher --ros-args -p image_source:=/workspace/Data/videos/07.Mediapipe/person_open_palm.avi
```

**화면 읽기:** /vla/image_overlay의 초록색 점은 전진·회전 중, 빨간색 점은 좌표가 있지만 정지 중입니다. 좌표가 없으면 NO TARGET입니다. Target은 거북이 목표 좌표, age는 제어에 사용한 프레임 경과 시간입니다. 승인 경과 시간·TTL·남은 유효시간도 확인합니다. VLM 추론 중에는 이전 관측 화면이 유지됩니다. 최신 원본은 /camera/image_raw입니다. mirror:=true는 목표 x만 반전하고 영상과 점은 원본 위치를 유지합니다.

**B — VLM 기반 이동:** 01의 B를 Ctrl+C로 종료합니다. 새 B에서도 실행 가능하도록 환경 설정을 포함했습니다. 공용 Ollama에 gemma4:e2b-it-qat가 있어야 합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=400 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/03-hand-vlm.jsonl
```

01과 같은 입력·초기 pose·mirror=false·TTL 400ms를 사용합니다. 여기서는 TTL을 바꾸지 않습니다.

**C — 관측·판단·행동 확인:** 새 Terminal에서 전체를 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 topic echo /vla/action
```

연속 출력은 Ctrl+C로 끝낸 뒤, 같은 C에서 필요에 따라 ros2 topic echo /vla/perception 또는 ros2 topic echo /vla/decision을 실행합니다. frame_id·reason·TTL·승인 경과 시간을 영상과 비교합니다.

**종료:** A 중단 → C에서 STOP 확인 → B 종료 → C 종료. 다음 노트북으로 바로 이어갈 때는 A를 유지하고 B만 종료합니다. GUI 없는 자습은 B launch 앞에 QT_QPA_PLATFORM=offscreen, 인자에 with_image_view:=false를 사용합니다.


### 실패 시나리오를 한 번에 비교
```bash
cd /workspace/09.Pseudo-VLA_v2
python scripts/run_hand_scenarios.py
```
실제 모델 호출 없이 같은 제어 함수에 실패 입력을 넣습니다. 출력 표에서 command와 reason을 함께 읽습니다.


## 6. 결과 해설과 다음 단계

모델 판단은 이동 제안입니다. 최종 행동은 관측의 신선도와 거북이 상태까지 검사한 제어기가 결정합니다.

고정 관측에 frame 불일치·승인 만료·pose 갱신 중단을 하나씩 넣습니다. 이후 동일 좌표·TTL에서 rule과 vlm의 움직임을 비교합니다.

VLM 추론 사이에 생기는 정지 구간을 이해하기 위해 다음에는 TTL과 대기 프레임을 분리해 봅니다.

비교표와 실제 화면·로그를 강사 설명과 대조합니다. 별도 답안 작성은 없으며, 자동 저장 파일과 실행 결과로 진행 상태를 확인합니다.
